# Whisper small zero-shot baseline on OpenSLR 104 test

This notebook runs `openai/whisper-small` zero-shot on the OpenSLR 104 Hinglish test set (2,942 clips, 4.76 h) and scores it with the project normalizer and metrics in `src/hinglish_asr/`. It runs in Google Colab on an L4 GPU.

- **Inputs:** the project folder on Google Drive, and `openslr104_test.zip` (the clips in `data/processed/openslr104/test/`).
- **Decoding:** language forced to Hindi, temperature fallback (0.0-1.0, compression ratio 1.35, logprob -1.0, seed 0), fp16. Clips over 30 s (3 clips) are split into 30 s chunks.
- **Outputs:** one row in `results/summary.csv`, and each clip's output in `results/whisper-small-zeroshot/hyps.jsonl`.

Note: with the current scoring code T-WER equals WER, because `Normalizer.normalize` converts Devanagari loanwords to English before `t_wer` runs.

## Setup

In [1]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/projects/asr"
ZIP_PATH = f"{PROJECT}/openslr104_test.zip"
AUDIO_DIR = "/content/openslr104_test"

Mounted at /content/drive


In [2]:
!pip install -q jiwer

In [3]:
import os
import sys

import torch

In [4]:
os.chdir(PROJECT)
sys.path.insert(0, f"{PROJECT}/src")
print(torch.cuda.get_device_name(0))

NVIDIA L4


In [5]:
import zipfile

from tqdm.auto import tqdm

with zipfile.ZipFile(ZIP_PATH) as z:
    for member in tqdm(z.infolist(), desc="unzip"):
        z.extract(member, AUDIO_DIR)

unzip:   0%|          | 0/5886 [00:00<?, ?it/s]

## Test manifest

The manifest's `audio_path` values are local paths on the machine that made the data, so each clip is found by file name in the unzipped folder.

In [6]:
import json
from pathlib import Path

wavs = {p.name: str(p) for p in Path(AUDIO_DIR).rglob("*.wav")}
with open("data/processed/openslr104/test_manifest.jsonl", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

missing = [r["utt_id"] for r in rows if Path(r["audio_path"]).name not in wavs]
assert not missing, f"{len(missing)} clips missing, e.g. {missing[:3]}"

paths = [wavs[Path(r["audio_path"]).name] for r in rows]
refs = [r["text"] for r in rows]
hours = sum(r["duration"] for r in rows) / 3600
print(f"{len(rows)} clips in manifest, {len(wavs)} wav files, {hours:.2f} h")
print(refs[0])
print(paths[0])

2942 clips in manifest, 5884 wav files, 4.76 h
libre office impress पर slide master और slide design के इस spoken tutorial में आपका स्वागत है
/content/openslr104_test/test/598753_406yMKxIdSDHRf8H_0000.wav


## Model

In [7]:
from transformers import pipeline

MODEL = "openai/whisper-small"
GENERATE = {"language": "hindi", "task": "transcribe", "num_beams": 1}

pipe = pipeline(
    "automatic-speech-recognition",
    model=MODEL,
    dtype=torch.float16,
    device="cuda:0",
    chunk_length_s=30,
)

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

... (4 outputs left out)

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


A check on 5 clips before the full run:

In [8]:
for path, ref in zip(paths[:5], refs[:5]):
    print("REF:", ref)
    print("HYP:", pipe(path, generate_kwargs=GENERATE)["text"], "\n")

REF: libre office impress पर slide master और slide design के इस spoken tutorial में आपका स्वागत है


[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


HYP:  लिबर अफिस इंप्रस पर यस याईद मास्टर और याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद याईद य 

REF: इस tutorial में हम यह सीखेंगे कि slides के लिए backgrounds layout कैसे लागू करें
HYP:  इस तीट्योल में हम सीकेंगे की, सलाइट्ट के लिए बैग्ग्डाूंँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँ 

REF: background slide पर लागू किये गये सभी रंगों और प्रभावों को उद्घृत करता है जो विषयवस्तु के पीछे मौजूद हैं
HYP:  बैक्ग्रून, स्लाइट पर लगु किये गय सभी रंगों और प्रभावों को उद्खित करता है, जो विश्यवस्टू के पीछे मोजुद है. 

REF: libre office impress में कई background options हैं जो बेहतर प्रस्तुतियाँ बनाने में आपकी मदद करते

## Note

There seems to be an issue with the decoding.

So basically the code above used greedy approach for decoding, which is to select the most probable token evrytime. Adding a temperature fallback can change that since it will stop the loops.

In [9]:
from transformers import pipeline, set_seed

MODEL = "openai/whisper-small"
GENERATE = {
    "language": "hindi",
    "task": "transcribe",
    "num_beams": 1,
    "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    "compression_ratio_threshold": 1.35,
    "logprob_threshold": -1.0,
}
set_seed(0)

pipe = pipeline(
    "automatic-speech-recognition",
    model=MODEL,
    dtype=torch.float16,
    device="cuda:0",
    chunk_length_s=30,
)

for path, ref in zip(paths[:5], refs[:5]):
    print("REF:", ref)
    print("HYP:", pipe(path, generate_kwargs=GENERATE)["text"], "\n")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


REF: libre office impress पर slide master और slide design के इस spoken tutorial में आपका स्वागत है
HYP:  लिबर अफिस इमप्रष पर श्लाइड मास्तर और श्लाइड ड़ाईगिर इस पोखें टिट्रल में अपका सवगत है. 

REF: इस tutorial में हम यह सीखेंगे कि slides के लिए backgrounds layout कैसे लागू करें
HYP:  नदाव मिसा कर सब थाव. की बाहब त्मिलनलननर पर ऻगात.. शुरिए कहाने करते हैं। और लेवना विहअने न काकल टरिका, ओह उन्या सबताश, और ज़ाने कराँगा. 

REF: background slide पर लागू किये गये सभी रंगों और प्रभावों को उद्घृत करता है जो विषयवस्तु के पीछे मौजूद हैं
HYP:  बैक्ग्रून, स्लाइट पर लगु किये गय सभी रंगों और प्रभावों को उद्खित करता है, जो विश्यवस्टू के पीछे मोजुद है. 

REF: libre office impress में कई background options हैं जो बेहतर प्रस्तुतियाँ बनाने में आपकी मदद करते हैं
HYP:  1. 

REF: आप भी अपने खुद़ के custom backgrounds बना सकते हैं
HYP:  अब भी अपने कुतके कस्तम बंगरूंँँस बना सकते हैं 



## Transcribe the test set

The input is a generator, so the pipeline returns results one at a time (in order).

In [10]:
import time

start = time.time()
outputs = pipe((p for p in paths), batch_size=32, generate_kwargs=GENERATE)
hyps = [out["text"] for out in tqdm(outputs, total=len(paths), desc="transcribe")]
elapsed = time.time() - start

assert len(hyps) == len(rows)
print(f"{len(hyps)} outputs, {sum(not h.strip() for h in hyps)} empty")
print(f"{elapsed / 60:.1f} min, real-time factor {elapsed / (hours * 3600):.4f}")

transcribe:   0%|          | 0/2942 [00:00<?, ?it/s]

2942 outputs, 0 empty
66.8 min, real-time factor 0.2339


Each clip's output is saved before scoring, so the clips can be re-scored later without the GPU.

In [11]:
out_dir = Path("results/whisper-small-zeroshot")
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / "hyps.jsonl", "w", encoding="utf-8") as f:
    for r, h in zip(rows, hyps):
        f.write(json.dumps({"utt_id": r["utt_id"], "ref": r["text"], "hyp": h}, ensure_ascii=False) + "\n")

## Score

In [12]:
from hinglish_asr.metrics import score
from hinglish_asr.utils import Normalizer

norm = Normalizer("data/lists/loanwords.csv")
assert score(refs, refs, norm) == {"wer": 0.0, "t_wer": 0.0}

scores = score(refs, hyps, norm)
print(f"WER {scores['wer']:.4f}  T-WER {scores['t_wer']:.4f}")

WER 0.9150  T-WER 0.9150


Normalized text that the metrics compare, for the first 5 clips:

In [13]:
for ref, hyp in zip(refs[:5], hyps[:5]):
    print("REF:", norm.for_scoring(ref))
    print("HYP:", norm.for_scoring(hyp), "\n")

REF: libre office impress पर slide master और slide design के इस spoken tutorial में आपका स्वागत है
HYP: libre अफिस इंप्रस पर श्लाइट मास्तर और श्लाइट डिशाह जिएदाराल ना अपका सवागत है 

REF: इस tutorial में हम यह सीखेंगे कि slides के लिए backgrounds layout कैसे लागू करें
HYP: भे ऄ जब चविर फ़रा के चखा कि प्र है प्र कल तब वो अगर्तांगता है थो खरू प्र खिल पोने सन्दॢ़क कि मीमल तबव्यवता णेट्या लिके प्र ड़चान प्र्षल्ख की ररठीठ मेंजा प्र कि अछी पाक वां बगॉ से हचाए लिए खिलत भी निहाएं की ि्र्बनिः चाने कन जिया के जिछां 

REF: background slide पर लागू किये गये सभी रंगों और प्रभावों को उद्घृत करता है जो विषयवस्तु के पीछे मौजूद हैं
HYP: बैक्ग्रून स्लाइट पर लगु किये गय सभी रंगों और प्रभावों को उद्खित करता है जो विश्यवस्टू के पीछे मोजुद है 

REF: libre office impress में कई background options हैं जो बेहतर प्रस्तुतियाँ बनाने में आपकी मदद करते हैं
HYP: libre अफिस इंप्रस में कई बैग्राँन अव्ष्ट्यन्स है जो बहतर प्रस्थुतिया बनाने में आपकी मदध्या 

REF: आप भी अपने खुद़ के custom backgrounds बना सकते हैं
HYP: आब

## Results file

In [14]:
import csv
import datetime

summary = Path("results/summary.csv")
row = {
    "model": MODEL,
    "dataset": "openslr104",
    "split": "test",
    "n_utts": len(rows),
    "wer": round(scores["wer"], 4),
    "t_wer": round(scores["t_wer"], 4),
    "decoding": "zero-shot, temperature fallback 0.0-1.0 (compression ratio 1.35, logprob -1.0, seed 0), language=hi, fp16, 30 s chunks",
    "date": datetime.date.today().isoformat(),
}
write_header = not summary.exists()
with open(summary, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if write_header:
        writer.writeheader()
    writer.writerow(row)

print(summary.read_text(encoding="utf-8"))

model,dataset,split,n_utts,wer,t_wer,decoding,date
openai/whisper-small,openslr104,test,2942,0.915,0.915,"zero-shot, greedy, language=hi, fp16, 30 s chunks",2026-11-23

